In [1]:
import pathlib 

import numpy as np
import scipy as sp
from matplotlib import pyplot as plt
import pickle

import morph_analyses as m

%matplotlib inline

%load_ext autoreload
%autoreload 2

In [7]:
df = m.preprocessing.load_session_db(dir= pathlib.Path(m.params.repo_dir) / 'data' / 'morph_vr_data')
df = df[df['RewardCount']>40]
df = df.sort_values(['MouseName','DateTime','SessionNumber'])
df = df[df["Track"]=="TwoTower_foraging"]

In [ ]:
def single_mouse_prior(
        mouse,df,
        first_ind=5,sigma_prior=.1,
        x = np.linspace(-.3,1.3,num=1000)[np.newaxis,:]):
    behav_sessions = df[df["MouseName"]==mouse]
    imag_sessions = behav_sessions[behav_sessions["Imaging"]==1]
    imag_sessions = imag_sessions[(imag_sessions['ImagingRegion']=='CA1' )|(imag_sessions['ImagingRegion']=='')]
    
    
    morph_data = single_mouse_data(mouse,behav_sessions)
    
    WALLMORPH,WALLMORPH_UC, RARE_YTARGET, FREQ_YTARGET,RARE_YHAT,FREQ_YHAT, PRIORS, PRIORS_UC,SF,SFHAT = [],[],[],[],[],[],[],[],[],[]
    
    for ind in range(first_ind,imag_sessions.shape[0]):
        sess = imag_sessions.iloc[ind]
        
        print(ind)
        # calculate cumulative prior up to current session
        date_folder,sessn = None,-np.inf
        em_all = None
        b_ind = 0
        while date_folder!=sess['DateFolder'] or sessn<sess["SessionNumber"]:
            b_sess = behav_sessions.iloc[b_ind]
            date_folder = b_sess['DateFolder']
            sessn = b_sess['SessionNumber']
            if em_all is None:
                em_all = morph_data[date_folder+str(sessn)]
            else:
                em_all = np.concatenate((em_all,morph_data[date_folder+str(sessn)]),axis=0)
            b_ind+=1


        em_1d = em_all[:,:1]
        prior = np.mean(u.gaussian(em_1d,sigma_prior,x),axis=0,keepdims=True)
        PRIORS.append(prior)
        prior_spl = lambda x: np.mean(u.gaussian(em_1d,sigma_prior,x),axis=0,keepdims=True)
        
        em_uc = em_all[:,1:2]
        prior_uc = np.mean(u.gaussian(em_uc,sigma_prior,x),axis=0,keepdims=True)
        PRIORS_UC.append(prior_uc)


